In [189]:
import sys
sys.path.append('..')

import os
from sqlalchemy import create_engine, text
import geopandas as gpd
import pandas as pd
#import georaster  # O usa rioxarray/rasterio si ya lo tienes abierto
import matplotlib.pyplot as plt

import ipywidgets as widgets

import numpy as np
from rasterio import features
from shapely.geometry import shape
from geoalchemy2 import Geometry, WKTElement
from shapely.geometry import Polygon, MultiPolygon

import rioxarray
import xarray as xr

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [174]:
def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

# get catastro desde base de datos
def get_catastro():
    engine = obtener_engine()
    try:
        query = f'''
            SELECT * FROM catastro_iag.catastro
        '''
        gdf = gpd.read_postgis(query, engine, geom_col='geom')
        return gdf
    except Exception as e:
        print(f"❌ Error al obtener la capa de catastro: {e}")
        return gpd.GeoDataFrame()
    return None

# get codigos de propiedad unicos desde base de datos de estimativas
def get_cod_props_estimativa():
    engine = obtener_engine()
    try:
        # Eliminamos la referencia a 'geom' ya que no se solicita en el SELECT
        query = "SELECT DISTINCT unidad_01 FROM catastro_iag.estimativa_historica"
        
        # Usamos pandas normal para leer datos alfanuméricos
        df = pd.read_sql(query, engine)
        return df
        
    except Exception as e:
        print(f"❌ Error al obtener los cod_propiedad de estimativas: {e}")
        return pd.DataFrame() # Retornamos un DataFrame vacío en caso de error

# visualisar el raster ndvi y catastro
def plot_superposicion_memoria(gdf, rds_ndvi):
    """
    Visualiza el catastro sobre el objeto NDVI que ya está en memoria.
    """
    # 1. Crear la figura y el eje
    fig, ax = plt.subplots(figsize=(15, 12))
    
    # 2. Capa de Fondo: NDVI
    # .squeeze() asegura que no haya dimensiones de banda extra
    # robust=True ajusta el contraste automáticamente
    rds_ndvi.squeeze().plot(
        ax=ax, 
        cmap='RdYlGn', 
        robust=True,
        cbar_kwargs={'label': 'Índice NDVI'}
    )
    
    # 3. Capa Superior: Catastro
    # facecolor='none' para ver el NDVI a través de los lotes
    gdf.plot(
        ax=ax, 
        edgecolor='black', 
        facecolor='none', 
        linewidth=0.2, 
        alpha=0.6
    )
    
    # 4. Ajustar el zoom a la extensión del catastro
    bounds = gdf.total_bounds
    ax.set_xlim([bounds[0], bounds[2]])
    ax.set_ylim([bounds[1], bounds[3]])
    
    ax.set_title("Verificación de Alineación: Catastro vs NDVI (UTM 20S)")
    plt.show()
    
# cargar raster NDVI y reproyectar a UTM
def cargar_y_reproyectar_ndvi(path, crs_destino):
    try:
        # Cargamos el raster
        rds = rioxarray.open_rasterio(path)
        
        # Reproyectamos a UTM 20S
        # 'nodata' asegura que los huecos de nubes se mantengan como nulos
        rds_utm = rds.rio.reproject(crs_destino)
        
        print(f"✅ Raster reproyectado exitosamente a {crs_destino}")
        print(f"Resolución en metros: {rds_utm.rio.resolution()}")
        
        return rds_utm
    except Exception as e:
        print(f"❌ Error al procesar el raster: {e}")
        return None

In [175]:
# 1. Rutas de archivos
ruta_ndvi = r'C:\Documents\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\ESTIMATIVAS\NDVI'
nombre_archivo = 'NDVI_Limpio_Para_Python_2026-02-01.tif'
path_completo = os.path.join(ruta_ndvi, nombre_archivo)
# 2. Definir el CRS de destino (UTM 20S)
crs_utm20s = "EPSG:32720"

In [176]:
gdf_catrastro = get_catastro()

In [177]:
# filtar caña habilitada para para zafra actual
gdf_catastro_canha = gdf_catrastro[(gdf_catrastro['unidad_03'] != 0) & (gdf_catrastro['cultivo'] == 'canha') & (gdf_catrastro['zafra'] == 2026)]
gdf_catastro_canha['area'].sum()

58710.5

In [178]:
# seleccionar columnas necesarias
gdf_catastro_canha = gdf_catastro_canha[['idd', 'geom', 'unidad_01', 'unidad_02', 'unidad_03', 'unidad_04', 'unidad_05', 'variedad', 'soca', 'area']]
gdf_catastro_canha.head(3)

,idd,geom,unidad_01,unidad_02,unidad_03,unidad_04,unidad_05,variedad,soca,area
2,10009,"MULTIPOLYGON (((506528.416 8095647.843, 506537...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L2,UCG9020,5.0,9.80
3,10080,"MULTIPOLYGON (((506006.724 8095047.828, 506003...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L4,UCG9020,3.0,12.80
4,15905,"MULTIPOLYGON (((506095.082 8094940.498, 506068...",32.0,RIO PAILON--ROCA GIL VICENTE,633.0,AGROP. VICENTE ROCA GIL SRL,L3,SP835073,6.0,10.61


In [20]:
# cargar NDVI en variable
ndvi_utm = cargar_y_reproyectar_ndvi(path_completo, crs_utm20s)

✅ Raster reproyectado exitosamente a EPSG:32720
Resolución en metros: (9.721636158788714, -9.721636158788714)


In [ ]:
# graficar NDVI y catastro en un plano
plot_superposicion_memoria(gdf_catastro_canha, ndvi_utm)

In [179]:
def procesar_lote_completo(row, rds_ndvi, fecha_analisis):
    """
    Segmenta el lote por categorías de NDVI y simula áreas faltantes por nubes.
    """
    # Definición de rangos fijos de NDVI
    RANGOS_NDVI = [-1, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 1.0]
    resultados = []
    
    try:
        # 1. Recorte del raster al lote
        lote_clip = rds_ndvi.rio.clip([row['geom']], from_disk=True)
        data = lote_clip.values.squeeze()
        
        # 2. Máscara de datos (donde no hay nubes)
        mask_datos = ~np.isnan(data)
        area_total_catastro = row['area'] # Usamos el área oficial del catastro
        
        # CASO A: Hay datos visibles en el lote
        if np.any(mask_datos):
            lote_clasificado = np.digitize(data, RANGOS_NDVI[1:-1]) + 1
            shapes = features.shapes(
                lote_clasificado.astype('int16'), 
                mask=mask_datos, 
                transform=lote_clip.rio.transform()
            )

            conteo_por_cat = {}
            # Guardamos los polígonos procesados
            for geom_shape, val in shapes:
                poly = shape(geom_shape).intersection(row['geom'])
                if not poly.is_empty:
                    cat = int(val)
                    area_seg = (poly.area / 10000)
                    conteo_por_cat[cat] = conteo_por_cat.get(cat, 0) + area_seg
                    
                    resultados.append({
                        'unidad_01': row['unidad_01'], 'unidad_02': row['unidad_02'],
                        'unidad_03': row['unidad_03'], 'unidad_04': row['unidad_04'],
                        'unidad_05': row['unidad_05'], 'variedad': row['variedad'],
                        'categoria': cat, 'area_ha': round(area_seg, 4),
                        'metodo_dato': 'PROCESADO', 'fecha_imagen': fecha_analisis,
                        'geom': poly
                    })

            # Lógica de Relleno para Nubes
            area_visible_total = sum(conteo_por_cat.values())
            if area_visible_total < (area_total_catastro * 0.99): # Si falta más del 1%
                area_faltante = area_total_catastro - area_visible_total
                for cat, area_v in conteo_por_cat.items():
                    proporcion = area_v / area_visible_total
                    # Creamos registro simulado (usamos geom del lote como referencia)
                    resultados.append({
                        **row.drop('geom').to_dict(),
                        'categoria': cat, 'area_ha': round(area_faltante * proporcion, 4),
                        'metodo_dato': 'SIMULADO', 'fecha_imagen': fecha_analisis,
                        'geom': row['geom'] 
                    })
        
        # CASO B: Lote 100% cubierto por nubes
        else:
            for cat in range(1, 9):
                resultados.append({
                    **row.drop('geom').to_dict(),
                    'categoria': cat, 'area_ha': round(area_total_catastro / 8, 4),
                    'metodo_dato': 'SIMULADO_TOTAL', 'fecha_imagen': fecha_analisis,
                    'geom': row['geom']
                })
                
    except Exception as e:
        print(f"Error procesando lote {row['unidad_05']}: {e}")
        
    return resultados

In [191]:
def cargar_datos_finales(gdf_resultados, config_db):
    engine = create_engine(
        f"postgresql://{config_db['USER']}:{config_db['PASSWORD']}@{config_db['HOST']}:{config_db['PORT']}/{config_db['DATABASE']}"
    )
    
    if 'geometry' in gdf_resultados.columns:
        gdf_resultados = gdf_resultados.rename(columns={'geometry': 'geom'})
    
    # --- CORRECCIÓN AQUÍ ---
    # Forzamos a que todo sea MultiPolygon
    gdf_resultados['geom'] = [
        MultiPolygon([feature]) if isinstance(feature, Polygon) else feature 
        for feature in gdf_resultados['geom']
    ]
    
    # Reparar y asegurar que el CRS sea correcto
    gdf_resultados['geom'] = gdf_resultados['geom'].buffer(0)
    
    try:
        gdf_resultados.to_postgis(
            name='estimativa_historica',
            con=engine,
            schema='catastro_iag',
            if_exists='append',
            index=False,
            # Cambiamos 'POLYGON' por 'MULTIPOLYGON'
            dtype={'geom': Geometry('MULTIPOLYGON', srid=32720)}
        )
        #print(f"✅ Carga completada: {len(gdf_resultados)} registros insertados como MultiPolygon.")
    except Exception as e:
        print(f"❌ Error en la carga: {e}")

In [183]:
gdf_catastro = gdf_catastro_canha.copy()
print('Cantidad de lotes:', len(gdf_catastro))

Cantidad de lotes: 10528


In [182]:
gdf_catastro = gdf_catastro_canha[gdf_catastro_canha['unidad_03'].isin([41594])]  # filtro
print('Cantidad de lotes:', len(gdf_catastro))

Cantidad de lotes: 112


In [184]:
# cantidad de propiedad para procesar
lis_codigos_catastro = list(set(gdf_catastro['unidad_01']))
lis_codigos_catastro = [int(i) for i in lis_codigos_catastro]
print('Cantidad de props en catastro:', len(lis_codigos_catastro))

Cantidad de props en catastro: 1702


In [185]:
# LISTA DE PROPIEDADES QUE ESTAN ESTIMATIVAS
cod_props_estimativas = get_cod_props_estimativa()
lista_codigos_props_en_estimativa = list(set(cod_props_estimativas['unidad_01']))
print('Cantidad de propiedades de estimativas: ', len(lista_codigos_props_en_estimativa))

Cantidad de propiedades de estimativas:  7


In [187]:
cod_props_faltantes = list(set(lis_codigos_catastro) - set(lista_codigos_props_en_estimativa))
print(cod_props_faltantes)

[1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 25, 26, 27, 28, 29, 31, 32, 33, 36, 38, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 50, 54, 55, 57, 61, 62, 63, 66, 68, 71, 72, 74, 76, 78, 79, 80, 83, 84, 85, 89, 92, 93, 94, 98, 100, 104, 105, 106, 108, 110, 111, 112, 113, 114, 116, 117, 118, 119, 122, 123, 124, 125, 126, 127, 129, 132, 133, 135, 136, 137, 139, 140, 142, 145, 147, 149, 150, 155, 159, 161, 162, 164, 168, 171, 172, 173, 174, 176, 178, 179, 182, 183, 184, 185, 186, 187, 188, 189, 195, 197, 198, 201, 205, 206, 209, 211, 212, 213, 216, 217, 218, 219, 220, 221, 222, 225, 226, 227, 228, 229, 230, 231, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 244, 246, 247, 249, 250, 251, 253, 256, 257, 258, 260, 261, 263, 264, 265, 267, 268, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 283, 284, 285, 286, 287, 288, 289, 290, 291, 292, 293, 294, 295, 297, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 309, 310, 311, 313, 314, 320, 324, 32

In [194]:
cod_props_faltantes = [1454, 1456, 1457, 1458, 1459, 1460, 1461, 1464, 1467, 1469, 1470, 1472, 1473, 1474, 1475, 1476, 1478, 1479, 1489, 1491, 1494, 1495, 1497, 1498, 1499, 1500, 1503, 1507, 1509, 1510, 1511, 1512, 1513, 1516, 1518, 1520, 1521, 1522, 1526, 1528, 1535, 1536, 1538, 1539, 1540, 1542, 1544, 1545, 1546, 1547, 1549, 1553, 1554, 1556, 1557, 1559, 1561, 1562, 1563, 1564, 1566, 1568, 1569, 1570, 1573, 1574, 1577, 1578, 1579, 1580, 1582, 1585, 1586, 1587, 1588, 1591, 1592, 1593, 1594, 1597, 1598, 1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1609, 1611, 1612, 1613, 1615, 1616, 1617, 1618, 1620, 1621, 1622, 1624, 1626, 1627, 1629, 1631, 1633, 1637, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646, 1648, 1649, 1650, 1652, 1653, 1654, 1655, 1656, 1658, 1659, 1660, 1661, 1662, 1663, 1665, 1667, 1668, 1669, 1670, 1671, 1672, 1673, 1674, 1675, 1676, 1677, 1682, 1683, 1684, 1685, 1686, 1687, 1688, 1689, 1690, 1694, 1695, 1696, 1699, 1700, 1701, 1702, 1703, 1704, 1706, 1707, 1709, 1710, 1711, 1712, 1714, 1716, 1719, 1720, 1721, 1723, 1724, 1726, 1730, 1731, 1732, 1733, 1735, 1736, 1737, 1738, 1739, 1740, 1741, 1744, 1745, 1748, 1750, 1756, 1757, 1758, 1759, 1760, 1761, 1762, 1763, 1764, 1765, 1766, 1767, 1768, 1769, 1770, 1771, 1773, 1774, 1775, 1776, 1777, 1778, 1783, 1788, 1790, 1791, 1793, 1794, 1795, 1797, 1798, 1799, 1802, 1805, 1806, 1808, 1812, 1815, 1816, 1817, 1818, 1819, 1820, 1823, 1824, 1826, 1827, 1828, 1829, 1830, 1831, 1832, 1833, 1844, 1845, 1846, 1847, 1848, 1849, 1850, 1851, 1852, 1853, 1854, 1855, 1856, 1857, 1861, 1862, 1865, 1867, 1868, 1869, 1870, 1871, 1873, 1874, 1875, 1876, 1878, 1879, 1880, 1881, 1882, 1883, 1884, 1885, 1886, 1887, 1888, 1889, 1892, 1893, 1895, 1898, 1899, 1900, 1901, 1902, 1904, 1905, 1906, 1907, 1908, 1909, 1910, 1911, 1912, 1915, 1916, 1917, 1918, 1919, 1922, 1925, 1926, 1930, 1931, 1932, 1933, 1934, 1935, 1936, 1937, 1938, 1939, 1941, 1944, 1947, 1948, 1949, 1950, 1951, 1953, 1955, 1958, 1959, 1960, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2022, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2045, 2046, 2047, 2048, 2049, 2051, 2052, 2053, 2054, 2055, 2056, 2057, 2058, 2059, 2060, 2061, 2062, 2063, 2066, 2067, 2068, 2069, 2070, 2071, 2072, 2073, 2074, 2075, 2076, 2077, 2078, 2079, 2080, 2081, 2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092, 2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2103, 2104, 2105, 2106, 2107, 2108, 2110, 2112, 2113, 2114, 2115, 2116, 2117, 2119, 2120, 2121, 2122, 2123, 2124, 2125, 2126, 2127, 2128, 2129, 2131, 2132, 2133, 2134, 2135, 2136, 2137, 2138, 2139, 2141, 2142, 2143, 2144, 2145, 2146, 2147, 2148, 2149, 2150, 2151, 2152, 2153, 2154, 2156, 2158, 2159, 2161, 2162, 2163, 2165, 2166, 2167, 2168, 2169, 2170, 2171, 2173, 2174, 2175, 2176, 2177, 2178, 2179, 2180, 2182, 2183, 2184, 2185, 2186, 2187, 2188, 2189, 2190, 2192, 2194, 2195, 2196, 2197, 2198, 2199, 2200, 2201, 2202, 2203, 2204, 2205, 2206, 2207, 2208, 2209, 2210, 2211, 2212, 2213, 2214, 2215, 2216, 2217, 2218, 2219, 2220, 2221, 2222, 2223, 2224, 2225, 2227, 2228, 2229, 2230, 2231, 2232, 2234, 2235, 2237, 2239, 2241, 2242, 2243, 2245, 2246, 2247, 2248, 2249, 2251, 2253, 2254, 2257, 2258, 2259, 2262, 2263, 2264, 2265, 2266, 2267, 2268, 2270, 2272, 2273, 2274, 2275, 2276, 2278, 2279, 2280, 2281, 2282, 2283, 2284, 2285, 2286, 2287, 2288, 2289, 2290, 2292, 2293, 2294, 2295, 2296, 2299, 2300, 2301, 2302, 2303, 2304, 2305, 2306, 2308, 2309, 2310, 2311, 2312, 2313, 2314, 2315, 2316, 2318, 2319, 2321, 2322, 2323, 2324, 2325, 2326, 2327, 2328, 2329, 2330, 2331, 2332, 2335, 2336, 2337, 2338, 2339, 2340, 2341, 2342, 2344, 2345, 2346, 2347, 2348, 2349, 2350, 2351, 2352, 2354, 2355, 2356, 2357, 2358, 2360, 2361, 2362, 2363, 2364, 2365, 2366, 2367]

In [196]:
estado_proceso = widgets.Output(layout={'border': '1px solid black'})
estado_proceso

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

In [197]:
# 2. Procesar todos los lotes
fecha_mosaico = '2026-02-01'
lista_final = []
df_final = None

for cod in cod_props_faltantes:
    df_final = None
    lista_final = []
    prop = gdf_catastro[gdf_catastro['unidad_01']==cod]
    #print("Iniciando análisis espacial propiedad: ", cod)
    for _, lote in prop.iterrows():
        res_lote = procesar_lote_completo(lote, ndvi_utm, fecha_mosaico)
        res_lote[0]
        lista_final.extend(res_lote)

    # 3. Convertir a GeoDataFrame especificando la columna geom
    df_final = gpd.GeoDataFrame(lista_final, geometry='geom', crs=32720)
    df_final['unidad_01'] = df_final['unidad_01'].fillna(0).astype(int)
    df_final['unidad_03'] = df_final['unidad_03'].fillna(0).astype(int)

    # 4. Cargar a la DB
    config_psql = POSTGRES_UTEA # Tu diccionario de configuración
    cargar_datos_finales(df_final, config_psql)
    
    with estado_proceso:
        estado_proceso.clear_output()
        display(f'ESTADO...')
        display(f'Se proceso propiedad: {cod}')

Error procesando lote L1.4: arr must be 2D or 3D array


IndexError: list index out of range

In [164]:
df_final

,unidad_01,unidad_02,unidad_03,unidad_04,unidad_05,variedad,categoria,area_ha,metodo_dato,fecha_imagen,geom
0,480,EL CANAL,41594,AGROPECUARIA CAMPO DULCE S.R.L.,Ca-5,UCG9020,3,0.0055,PROCESADO,2026-02-01,"POLYGON ((518628.679 8100117.351, 518618.957 8..."
1,480,EL CANAL,41594,AGROPECUARIA CAMPO DULCE S.R.L.,Ca-5,UCG9020,4,0.0052,PROCESADO,2026-02-01,"POLYGON ((518570.349 8100107.630, 518560.628 8..."
2,480,EL CANAL,41594,AGROPECUARIA CAMPO DULCE S.R.L.,Ca-5,UCG9020,4,0.0769,PROCESADO,2026-02-01,"POLYGON ((518628.679 8100117.351, 518628.679 8..."
3,480,EL CANAL,41594,AGROPECUARIA CAMPO DULCE S.R.L.,Ca-5,UCG9020,3,0.0048,PROCESADO,2026-02-01,"POLYGON ((518512.019 8100097.908, 518502.298 8..."
4,480,EL CANAL,41594,AGROPECUARIA CAMPO DULCE S.R.L.,Ca-5,UCG9020,4,0.0240,PROCESADO,2026-02-01,"POLYGON ((518541.184 8100107.630, 518541.184 8..."
...,...,...,...,...,...,...,...,...,...,...,...
20422,2238,EL PARAISO,41594,AGROPECUARIA CAMPO DULCE S.R.L.,L22.2,CITTCA0563,8,0.0404,PROCESADO,2026-02-01,"POLYGON ((517014.887 8070223.320, 517112.104 8..."
20423,2238,EL PARAISO,41594,AGROPECUARIA CAMPO DULCE S.R.L.,L22.2,CITTCA0563,6,0.0098,PROCESADO,2026-02-01,"POLYGON ((517112.104 8070223.320, 517131.547 8..."
20424,2238,EL PARAISO,41594,AGROPECUARIA CAMPO DULCE S.R.L.,L22.2,CITTCA0563,6,0.1424,PROCESADO,2026-02-01,"POLYGON ((517160.712 8070223.320, 517180.155 8..."
20425,2238,EL PARAISO,41594,AGROPECUARIA CAMPO DULCE S.R.L.,L22.2,CITTCA0563,7,0.0191,PROCESADO,2026-02-01,"POLYGON ((517209.320 8070223.320, 517238.485 8..."
